In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab2") \
    .config("spark.jars.packages", "com.databricks:spark-xml_2.12:0.16.0") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.1


# Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы.

In [9]:
import os
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Получаем полный путь к файлам
current_dir = os.getcwd()
posts_path = os.path.join(current_dir, "posts_sample.xml")
languages_path = os.path.join(current_dir, "programming-languages.csv")

print("Posts path:", posts_path)
print("Languages path:", languages_path)

# Читаем XML с полным путем
posts_df = spark.read \
    .format("xml") \
    .option("rowTag", "row") \
    .option("inferSchema", "true") \
    .load(posts_path)

# Читаем языки
languages_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(languages_path)

print("Posts loaded:", posts_df.count())
print("Languages loaded:", languages_df.count())

# Обработка
posts_with_year = posts_df.withColumn("year", F.year("_CreationDate"))
filtered_posts = posts_with_year.filter((F.col("year") >= 2010) & (F.col("year") <= 2020))

posts_with_tags = filtered_posts.filter(F.col("_Tags").isNotNull())
tags_processed = posts_with_tags.withColumn(
    "tags_array",
    F.split(F.regexp_replace(F.col("_Tags"), "^<|>$", ""), "><")
)
exploded_tags = tags_processed.select("year", F.explode("tags_array").alias("tag")).filter(F.col("tag") != "")

languages_list = [row.name.lower() for row in languages_df.select("name").collect()]
programming_langs = exploded_tags.filter(F.lower(F.col("tag")).isin(languages_list))

yearly_counts = programming_langs.groupBy("year", "tag").agg(F.count("*").alias("mention_count"))

window_spec = Window.partitionBy("year").orderBy(F.desc("mention_count"))
top10_by_year = yearly_counts \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") <= 10) \
    .orderBy("year", "rank")

print("Data processed")
print("Total records:", top10_by_year.count())
top10_by_year.show(50, truncate=False)

Posts path: /content/LR2/posts_sample.xml
Languages path: /content/LR2/programming-languages.csv
Posts loaded: 46006
Languages loaded: 700
Data processed
Total records: 100
+----+-----------+-------------+----+
|year|tag        |mention_count|rank|
+----+-----------+-------------+----+
|2010|java       |52           |1   |
|2010|php        |46           |2   |
|2010|javascript |44           |3   |
|2010|python     |26           |4   |
|2010|objective-c|23           |5   |
|2010|c          |20           |6   |
|2010|ruby       |12           |7   |
|2010|delphi     |8            |8   |
|2010|applescript|3            |9   |
|2010|r          |3            |10  |
|2011|php        |102          |1   |
|2011|java       |93           |2   |
|2011|javascript |83           |3   |
|2011|python     |37           |4   |
|2011|objective-c|34           |5   |
|2011|c          |24           |6   |
|2011|ruby       |20           |7   |
|2011|perl       |9            |8   |
|2011|delphi     |8          

# Получившийся отчёт сохранить в формате Apache Parquet.

In [11]:
def get_path_to_file(file_name):
    return f'file://{os.getcwd()}/{file_name}'.replace("\\", "/")

output_path = get_path_to_file("top_ten_languages.parquet")
print("Saving to:", output_path)

top10_by_year.write.mode("overwrite").parquet(output_path)
print("Parquet saved successfully")

saved_data = spark.read.parquet(output_path)
print("Total records:", saved_data.count())
saved_data.show(10)

Saving to: file:///content/LR2/top_ten_languages.parquet
Parquet saved successfully
Total records: 100
+----+-----------+-------------+----+
|year|        tag|mention_count|rank|
+----+-----------+-------------+----+
|2010|       java|           52|   1|
|2010|        php|           46|   2|
|2010| javascript|           44|   3|
|2010|     python|           26|   4|
|2010|objective-c|           23|   5|
|2010|          c|           20|   6|
|2010|       ruby|           12|   7|
|2010|     delphi|            8|   8|
|2010|applescript|            3|   9|
|2010|          r|            3|  10|
+----+-----------+-------------+----+
only showing top 10 rows



In [12]:
from google.colab import files

!zip -r top_ten_languages.parquet.zip top_ten_languages.parquet
files.download("top_ten_languages.parquet.zip")

  adding: top_ten_languages.parquet/ (stored 0%)
  adding: top_ten_languages.parquet/.part-00000-8e3fea93-dd91-497d-a17d-a74079901b6f-c000.snappy.parquet.crc (stored 0%)
  adding: top_ten_languages.parquet/._SUCCESS.crc (stored 0%)
  adding: top_ten_languages.parquet/part-00000-8e3fea93-dd91-497d-a17d-a74079901b6f-c000.snappy.parquet (deflated 38%)
  adding: top_ten_languages.parquet/_SUCCESS (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>